# Ch7 Playground — 點積與對偶性

> 對應 3Blue1Brown《線性代數的本質》第七章
>
> **核心大問**：點積只是「對應元素相乘再相加」嗎？  
> 為什麼一個純粹的計算規則，跟「投影」和「角度」有這麼深的幾何關係？

---

## 本章路線圖

| Part | 主題 | 你會體會到的事 |
|---|---|---|
| 1 | 點積的計算與幾何意義 | $\vec{a} \cdot \vec{b} = \|\vec{a}\|\|\vec{b}\|\cos\theta$ 不只是公式，是投影 |
| 2 | 投影 (Projection) | 把一個向量「投影」到另一個向量上，長度就是點積的幾何來源 |
| 3 | 點積的正負零 | 正 = 同方向、負 = 反方向、零 = 垂直 |
| 4 | 對偶性 (Duality) | **本章最關鍵的洞見**：1×2 矩陣 ↔ 2D 向量，點積其實是線性變換 |
| 5 | 對偶性的視覺證明 | 為什麼「數字線上的投影」和「1×2 矩陣乘法」是同一件事 |
| 6 | 動手實驗區 | 自己玩 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 7)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 11

def setup_ax(ax, lim=4, title=""):
    """統一設定座標軸"""
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    if title: ax.set_title(title, fontsize=13)

def draw_vec(ax, v, color='blue', label='', lw=2.5):
    """畫一個從原點出發的向量箭頭"""
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        offset = v / np.linalg.norm(v) * 0.3
        ax.text(v[0]+offset[0], v[1]+offset[1], label, fontsize=12,
                fontweight='bold', color=color, ha='center')

def draw_angle_arc(ax, a, b, radius=0.6):
    """畫兩向量之間的角度弧"""
    angle_a = np.degrees(np.arctan2(a[1], a[0]))
    angle_b = np.degrees(np.arctan2(b[1], b[0]))
    if angle_a > angle_b:
        angle_a, angle_b = angle_b, angle_a
    arc = Arc((0,0), 2*radius, 2*radius, angle=0,
              theta1=angle_a, theta2=angle_b, color='gray', lw=1.5)
    ax.add_patch(arc)
    mid_angle = np.radians((angle_a + angle_b) / 2)
    ax.text(radius*1.3*np.cos(mid_angle), radius*1.3*np.sin(mid_angle),
            'θ', fontsize=14, color='gray', ha='center', va='center')

print("工具函式準備完成！")

---
## Part 1：點積 — 兩種看法

### 計算定義（代數）

$$\vec{a} \cdot \vec{b} = a_1 b_1 + a_2 b_2$$

就是「對應元素相乘再相加」。但這給你零直覺。

### 幾何定義

$$\vec{a} \cdot \vec{b} = \|\vec{a}\| \cdot \|\vec{b}\| \cdot \cos\theta$$

其中 $\theta$ 是兩向量之間的夾角。

**直覺**：把 $\vec{b}$ 投影到 $\vec{a}$ 的方向上，得到投影長度，再乘以 $\|\vec{a}\|$。
所以點積衡量的是「$\vec{b}$ 在 $\vec{a}$ 方向上走了多遠，再用 $\vec{a}$ 的長度加權」。

In [ ]:
# === 實驗 1：兩種算法得到同一個結果 ===

a = np.array([3, 1])
b = np.array([1, 2])

# 方法 1：代數（對應元素相乘再加）
dot_algebra = np.dot(a, b)    # 也可以寫 a @ b 或 a.dot(b)
print(f"代數：a·b = {a[0]}×{b[0]} + {a[1]}×{b[1]} = {dot_algebra}")

# 方法 2：幾何（|a| × |b| × cosθ）
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
cos_theta = np.dot(a, b) / (norm_a * norm_b)
theta = np.arccos(np.clip(cos_theta, -1, 1))

dot_geometry = norm_a * norm_b * np.cos(theta)
print(f"幾何：|a|×|b|×cosθ = {norm_a:.3f} × {norm_b:.3f} × cos({np.degrees(theta):.1f}°) = {dot_geometry:.4f}")
print(f"\n兩者一致？{np.isclose(dot_algebra, dot_geometry)}")

# 視覺化
fig, ax = plt.subplots(figsize=(7, 7))
draw_vec(ax, a, 'red', f'a = ({a[0]},{a[1]})')
draw_vec(ax, b, 'blue', f'b = ({b[0]},{b[1]})')
draw_angle_arc(ax, a, b)
setup_ax(ax, lim=4, title=f"a · b = {dot_algebra}\nθ = {np.degrees(theta):.1f}°")
plt.show()

---
## Part 2：投影 (Projection) — 點積的幾何核心

### 理論

把 $\vec{b}$ 投影到 $\vec{a}$ 上：

$$\text{proj}_{\vec{a}} \vec{b} = \frac{\vec{a} \cdot \vec{b}}{\vec{a} \cdot \vec{a}} \cdot \vec{a}$$

拆解來看：
1. $\frac{\vec{a} \cdot \vec{b}}{|\vec{a}|^2}$ 是一個**純量（scalar）**，代表「$\vec{b}$ 在 $\vec{a}$ 方向上走了幾個 $\vec{a}$ 的長度」
2. 乘上 $\vec{a}$ 就得到投影向量

投影長度（帶正負號）：
$$\text{scalar projection} = \frac{\vec{a} \cdot \vec{b}}{|\vec{a}|} = |\vec{b}| \cos\theta$$

In [ ]:
# === 實驗 2：投影的視覺化 ===

def plot_projection(a, b, ax=None):
    """視覺化 b 投影到 a 上的完整過程"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    
    # 計算投影
    scalar_proj = np.dot(a, b) / np.dot(a, a)  # b 在 a 方向上的係數
    proj = scalar_proj * a                       # 投影向量
    residual = b - proj                          # 殘差（垂直分量）
    
    # 畫 a 的延長線（淺色）
    t = np.linspace(-2, 3, 100)
    a_hat = a / np.linalg.norm(a)
    ax.plot(t * a_hat[0] * np.linalg.norm(a), 
            t * a_hat[1] * np.linalg.norm(a), 
            'r-', alpha=0.15, lw=8)
    
    # 畫向量
    draw_vec(ax, a, 'red', 'a')
    draw_vec(ax, b, 'blue', 'b')
    draw_vec(ax, proj, 'darkred', 'proj')
    
    # 畫 b 的端點到投影點的虛線（垂直分量）
    ax.plot([b[0], proj[0]], [b[1], proj[1]], 'g--', lw=1.5, alpha=0.7)
    
    # 畫直角符號
    if np.linalg.norm(residual) > 0.1 and np.linalg.norm(proj) > 0.1:
        corner_size = 0.2
        perp_dir = residual / np.linalg.norm(residual)
        proj_dir = proj / np.linalg.norm(proj)
        corner = proj + corner_size * perp_dir
        p1 = proj + corner_size * perp_dir
        p2 = p1 + corner_size * proj_dir * (-1 if scalar_proj < 0 else -1)
        p3 = proj + corner_size * proj_dir * (-1 if scalar_proj < 0 else -1)
        # Simple right angle marker
        ax.plot([p1[0], p1[0]-corner_size*proj_dir[0]], 
                [p1[1], p1[1]-corner_size*proj_dir[1]], 'g-', lw=1)
        ax.plot([p3[0], p3[0]+corner_size*perp_dir[0]], 
                [p3[1], p3[1]+corner_size*perp_dir[1]], 'g-', lw=1)
    
    draw_angle_arc(ax, a, b)
    
    # 標注數值
    dot_val = np.dot(a, b)
    proj_length = dot_val / np.linalg.norm(a)
    theta = np.degrees(np.arccos(np.clip(
        dot_val / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1)))
    
    info = (f"a·b = {dot_val:.2f}\n"
            f"|b|cosθ = {proj_length:.2f} (scalar proj)\n"
            f"θ = {theta:.1f}°")
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=11,
            va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    setup_ax(ax, lim=4)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Case 1：銳角 → 投影在 a 的同側
plot_projection([3, 1], [1, 2], ax=axes[0])
axes[0].set_title("Case 1: θ < 90° (銳角)\nProjection 在 a 的同方向", fontsize=12)

# Case 2：鈍角 → 投影在 a 的反側
plot_projection([3, 1], [-1, 2], ax=axes[1])
axes[1].set_title("Case 2: θ > 90° (鈍角)\nProjection 在 a 的反方向", fontsize=12)

# Case 3：直角 → 投影 = 0
plot_projection([3, 1], [-1, 3], ax=axes[2])
axes[2].set_title("Case 3: θ = 90° (直角)\nProjection = 0 !", fontsize=12)

plt.tight_layout()
plt.show()

---
## Part 3：點積的正負零 — 方向感測器

### 理論

點積是最好的「方向探測器」：

| a · b | 幾何意義 | θ 的範圍 |
|---|---|---|
| > 0 | 大致同方向 | 0° ≤ θ < 90° |
| = 0 | 垂直（正交） | θ = 90° |
| < 0 | 大致反方向 | 90° < θ ≤ 180° |

這在機器學習中非常重要 — cosine similarity（餘弦相似度）就是把兩個向量正規化後取點積，用來衡量「方向有多像」。

In [ ]:
# === 實驗 3：360° 掃描 — 點積如何隨角度變化 ===

a = np.array([1, 0])  # 固定 a 指向右方

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：b 繞一圈，用顏色表示點積的正負
ax = axes[0]
angles = np.linspace(0, 2*np.pi, 200)
for ang in angles:
    b = np.array([np.cos(ang), np.sin(ang)])
    dot = np.dot(a, b)
    color = 'red' if dot > 0.01 else ('blue' if dot < -0.01 else 'green')
    ax.scatter(b[0], b[1], c=color, s=15, alpha=0.7)

draw_vec(ax, a * 1.3, 'black', 'a', lw=3)
ax.text(0.5, 1.15, 'dot > 0', color='red', fontsize=12, fontweight='bold')
ax.text(0.5, -1.15, 'dot > 0', color='red', fontsize=12, fontweight='bold')
ax.text(-1.3, 0.5, 'dot < 0', color='blue', fontsize=12, fontweight='bold')
ax.text(0.05, 1.15, 'dot=0', color='green', fontsize=11, fontweight='bold')
ax.text(0.05, -1.15, 'dot=0', color='green', fontsize=11, fontweight='bold')
setup_ax(ax, lim=1.8, title="b 繞單位圓一圈\n紅=正 藍=負 綠=零（垂直）")

# 右圖：點積值 vs 角度的曲線
ax = axes[1]
degs = np.linspace(0, 360, 361)
dots = [np.dot(a, [np.cos(np.radians(d)), np.sin(np.radians(d))]) for d in degs]
ax.plot(degs, dots, 'k-', lw=2)
ax.fill_between(degs, dots, 0, where=[d > 0 for d in dots], alpha=0.3, color='red', label='dot > 0')
ax.fill_between(degs, dots, 0, where=[d < 0 for d in dots], alpha=0.3, color='blue', label='dot < 0')
ax.axhline(0, color='green', lw=2, ls='--')
ax.set_xlabel('θ (degrees)', fontsize=12)
ax.set_ylabel('a · b', fontsize=12)
ax.set_title("點積 = cos(θ)\n（當 a, b 都是單位向量時）", fontsize=13)
ax.set_xticks([0, 90, 180, 270, 360])
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()
print("點積就是 cosine！當兩向量都是單位向量時，dot product = cos(θ)")
print("這就是為什麼 ML 中用 cosine similarity 來衡量向量相似度")

---
## Part 4：對偶性 (Duality) — 本章的靈魂

### 理論：3Blue1Brown 的核心洞見

這裡是整個第七章最重要的一段。

考慮一個 **1×2 矩陣**（從 2D 映射到 1D 數字線的線性變換）：

$$\begin{bmatrix} u_1 & u_2 \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = u_1 x + u_2 y$$

等等……$u_1 x + u_2 y$ 不就是向量 $(u_1, u_2)$ 和向量 $(x, y)$ 的**點積**嗎？！

$$\vec{u} \cdot \vec{v} = \begin{bmatrix} u_1 & u_2 \end{bmatrix} \vec{v}$$

**對偶性就是**：每一個 2D 向量 $\vec{u}$，都可以「對偶地」對應到一個 1×2 的線性變換（把 2D 空間投射到數字線上）。反之亦然。

> 「向量」和「把空間投射到數字線的線性變換」是**同一件事的兩種面貌**。

In [ ]:
# === 實驗 4a：向量 ↔ 1��2 矩陣，計算完全相同 ===

u = np.array([2, 1])         # 一個 2D 向量
M = np.array([[2, 1]])       # 對應的 1×2 矩陣

v = np.array([3, -1])        # 某個輸入

# 方法 1：點積
result_dot = np.dot(u, v)

# 方法 2：1×2 矩陣乘法
result_mat = (M @ v)[0]      # M @ v 結果是 [5]，取出純量

print(f"向量 u = {u}")
print(f"矩陣 M = {M}")
print(f"輸入 v = {v}")
print()
print(f"點積：  u · v   = {u[0]}×{v[0]} + {u[1]}×{v[1]} = {result_dot}")
print(f"矩陣：  M @ v   = {M[0,0]}×{v[0]} + {M[0,1]}×{v[1]} = {result_mat}")
print(f"\n完全一樣！這不是巧合 — 這就是對偶性。")

---
## Part 5：對偶性的視覺證明 — 「投影到數字線」

### 理論

3Blue1Brown 的論證方式：

1. 想像把一條**數字線**斜斜地放在 2D 空間裡，讓它通過原點，方向是 $\hat{u}$
2. 把所有 2D 向量**投影**到這條數字線上 → 這是一個從 2D 到 1D 的線性變換
3. 這個線性變換可以用一個 1×2 矩陣表示
4. 這個 1×2 矩陣的兩個元素……恰好就是 $\hat{u}$ 的座標！
5. 所以：**跟 $\hat{u}$ 做點積 = 投影到 $\hat{u}$ 方向的數字線上**

下面我們來視覺化這個過程。

In [ ]:
# === 實驗 5a：1×2 矩陣把 2D 壓到數字線 ===

def plot_duality(u_vec, test_points=None, figsize=(18, 6)):
    """
    視覺化對偶性：向量 u 既是 2D 空間中的箭頭，
    也是一個把 2D 投射到數字線的線性變換。
    """
    u = np.array(u_vec, dtype=float)
    u_hat = u / np.linalg.norm(u)
    
    if test_points is None:
        test_points = [[1, 0], [0, 1], [1, 1], [-1, 1], [2, 0.5], [-0.5, 1.5]]
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # === 左圖：2D 空間中的向量 u 和測試點 ===
    ax = axes[0]
    
    # 畫 u 方向的數字線
    t_line = np.linspace(-4, 4, 100)
    ax.plot(t_line * u_hat[0], t_line * u_hat[1], 'gray', lw=1, alpha=0.5)
    
    # 畫 u
    draw_vec(ax, u, 'red', f'u = ({u[0]:.0f},{u[1]:.0f})')
    
    # 畫測試點和投影
    colors = plt.cm.Set2(np.linspace(0, 1, len(test_points)))
    for i, p in enumerate(test_points):
        p = np.array(p, dtype=float)
        proj_scalar = np.dot(u_hat, p)
        proj_pt = proj_scalar * u_hat
        
        ax.scatter(*p, c=[colors[i]], s=60, zorder=5, edgecolors='black')
        ax.plot([p[0], proj_pt[0]], [p[1], proj_pt[1]], 
                color=colors[i], ls='--', lw=1, alpha=0.6)
        ax.scatter(*proj_pt, c=[colors[i]], s=40, zorder=5, marker='s')
    
    setup_ax(ax, lim=3, title="2D 空間\n(點 → 投影到 u 的方向)")
    
    # === 中圖：數字線上的結果 ===
    ax = axes[1]
    ax.axhline(0, color='gray', lw=2)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlim(-5, 5)
    
    # 畫刻度
    for tick in range(-4, 5):
        ax.plot([tick, tick], [-0.05, 0.05], 'k-', lw=1)
        ax.text(tick, -0.15, str(tick), ha='center', fontsize=9)
    
    for i, p in enumerate(test_points):
        p = np.array(p, dtype=float)
        # 方法 1：投影（幾何）
        proj_val = np.dot(u, p)  # 注意：用 u 而不是 u_hat，因為點積是和完整 u 做
        ax.scatter(proj_val, 0, c=[colors[i]], s=80, zorder=5, edgecolors='black')
        ax.text(proj_val, 0.12 + (i%2)*0.1, f'({p[0]:.0f},{p[1]:.0f})→{proj_val:.1f}',
                fontsize=9, ha='center', color=colors[i])
    
    ax.set_title(f"數字線上的結果\nu · v = [1×2 matrix] @ v", fontsize=12)
    ax.set_yticks([])
    
    # === 右圖：驗證 — dot product vs matrix multiplication ===
    ax = axes[2]
    M = u.reshape(1, 2)  # 1×2 矩���
    
    dot_results = []
    mat_results = []
    labels = []
    for p in test_points:
        p = np.array(p, dtype=float)
        dot_results.append(np.dot(u, p))
        mat_results.append((M @ p)[0])
        labels.append(f'({p[0]:.0f},{p[1]:.0f})')
    
    x_pos = np.arange(len(test_points))
    width = 0.35
    ax.bar(x_pos - width/2, dot_results, width, label='dot(u, v)', color='steelblue', alpha=0.8)
    ax.bar(x_pos + width/2, mat_results, width, label='M @ v', color='salmon', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Result')
    ax.legend(fontsize=11)
    ax.set_title(f"驗證：dot(u, v) ≡ [u₁ u₂] @ v\nM = [{u[0]:.0f} {u[1]:.0f}]", fontsize=12)
    ax.axhline(0, color='k', lw=0.5)
    
    plt.tight_layout()
    plt.show()

# 用 u = (2, 1) 來展示
plot_duality([2, 1])
print("左圖：2D 空間中的點被投影到 u 的方向上")
print("中圖：投影結果落在數字線上的位置")
print("右圖：dot product 和 1×2 matrix 乘法的結果完全相同！")

In [ ]:
# === 實驗 5b：為什麼 1×2 矩陣的元素就是 û 的座標？===
# 
# 線性變換完全由「基底去了哪裡」決定。
# 一個從 2D → 1D 的線性變換，只需要知道：
#   - î = (1,0) 被送到數字線上的哪裡？
#   - ĵ = (0,1) 被送到數字線上的哪裡？
#
# 如果這個變換是「投影到 û 方向」：
#   - î 投影到 û 上的位置 = û · î = u₁  （就是 û 的 x 分量）
#   - ĵ 投影到 û 上的位置 = û · ĵ = u₂  （就是 û 的 y 分量）
#
# 所以 1×2 矩陣 = [u₁  u₂] 就是 û 本身！

u_hat = np.array([2, 1]) / np.linalg.norm([2, 1])
i_hat = np.array([1, 0])
j_hat = np.array([0, 1])

print("û (單位向量) =", np.round(u_hat, 4))
print()
print("î = (1,0) 投影到 û 方向:")
print(f"  û · î = {np.dot(u_hat, i_hat):.4f}  ← 就是 û 的 x 分量！")
print()
print("ĵ = (0,1) 投影到 û 方向:")
print(f"  û · ĵ = {np.dot(u_hat, j_hat):.4f}  ← 就是 û 的 y 分量！")
print()
print(f"所以 1×2 矩陣 = [{u_hat[0]:.4f}  {u_hat[1]:.4f}]")
print(f"這正好就是 û 本身的座標！")
print()
print("═══════════════════════════════════════════════")
print("結論：向量的座標 ↔ 對應線性變換的矩陣表示")
print("      這就是 DUALITY（對偶性）")
print("═══════════════════════════════════════════════")

In [ ]:
# === 實驗 5c：視覺化基底向量的投影 → 推導出 1×2 矩陣 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

u = np.array([2, 1], dtype=float)
u_hat = u / np.linalg.norm(u)

for idx, (basis, name, color) in enumerate([(np.array([1.,0.]), 'î', 'red'), 
                                              (np.array([0.,1.]), 'ĵ', 'green')]):
    ax = axes[idx]
    
    # 畫 u 方向的數字線
    t = np.linspace(-3, 3, 100)
    ax.plot(t * u_hat[0], t * u_hat[1], 'gray', lw=6, alpha=0.2)
    
    # 在數字線上畫刻度
    for tick in np.arange(-2, 3, 0.5):
        pt = tick * u_hat
        perp = np.array([-u_hat[1], u_hat[0]])
        ax.plot([pt[0]-0.05*perp[0], pt[0]+0.05*perp[0]],
                [pt[1]-0.05*perp[1], pt[1]+0.05*perp[1]], 'gray', lw=1)
    
    # 畫 u 和基底向量
    draw_vec(ax, u, 'purple', f'u = ({u[0]:.0f},{u[1]:.0f})', lw=2)
    draw_vec(ax, basis, color, name, lw=3)
    
    # 投影
    proj_scalar = np.dot(u_hat, basis)  # 投到 û 方向的值
    proj_pt = proj_scalar * u_hat
    
    ax.plot([basis[0], proj_pt[0]], [basis[1], proj_pt[1]], 
            'k--', lw=1.5, alpha=0.5)
    ax.scatter(*proj_pt, c='black', s=100, zorder=10, marker='D')
    ax.annotate(f'proj = {proj_scalar:.3f}', xy=proj_pt,
                fontsize=12, fontweight='bold',
                xytext=(15, 15), textcoords='offset points',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    setup_ax(ax, lim=2.5, title=f"{name} 投影到 û 方向\n→ 1×2 矩陣的第 {idx+1} 個元素")

plt.tight_layout()
plt.show()

print(f"û · î = {np.dot(u_hat, [1,0]):.4f}  → 矩陣第 1 個元素")
print(f"û · ĵ = {np.dot(u_hat, [0,1]):.4f}  → 矩陣第 2 個元素")
print(f"1×2 矩陣 = [{np.dot(u_hat, [1,0]):.4f}  {np.dot(u_hat, [0,1]):.4f}]")
print(f"û 本身   = [{u_hat[0]:.4f}  {u_hat[1]:.4f}]")
print(f"完全一樣！這就是線性變換 ↔ 向量的對偶。")

---
## Part 6：動手實驗區

### 練習 A：改 u 的方向
修改下面的 `u` 向量，觀察：
1. u 越長，投影結果（數字線上的值）如何被放大？
2. u 轉到不同角度，哪些測試點的投影會從正變負？

### 練習 B：cosine similarity
在 ML 中，兩個向量的「方向相似度」定義為：
$$\text{cosine similarity} = \frac{\vec{a} \cdot \vec{b}}{|\vec{a}||\vec{b}|}$$
範圍 [-1, 1]。試試看用下面的工具計算。

In [ ]:
# === 練習 A：修改 u，觀察對偶性視覺化的變化 ===

u = [1, 2]   # ← 改這裡！試試 [3, 0], [0, 2], [-1, 1], [1, -1]

plot_duality(u)

In [ ]:
# === 練習 B：Cosine Similarity 計算器 ===

def cosine_sim(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 想像這是兩個 word embedding 向量
word_king  = [2, 3]
word_queen = [2.1, 2.8]
word_car   = [-1, 2]

pairs = [
    ("king", word_king, "queen", word_queen),
    ("king", word_king, "car", word_car),
    ("queen", word_queen, "car", word_car),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name_a, va, name_b, vb) in enumerate(pairs):
    ax = axes[idx]
    va, vb = np.array(va), np.array(vb)
    sim = cosine_sim(va, vb)
    
    draw_vec(ax, va, 'red', name_a)
    draw_vec(ax, vb, 'blue', name_b)
    draw_angle_arc(ax, va, vb)
    
    setup_ax(ax, lim=4, title=f"cosine_sim = {sim:.4f}")

plt.tight_layout()
plt.show()

print("Cosine similarity:")
for name_a, va, name_b, vb in pairs:
    sim = cosine_sim(va, vb)
    print(f"  {name_a} vs {name_b}: {sim:.4f}")
print()
print("king 和 queen 方向幾乎一樣 (similarity ≈ 1)")
print("king 和 car 方向差很多 (similarity 低)")
print("→ 這就是 NLP 中 word embedding 的核心思想！")

---
## 總結：點積的三層理解

| 層次 | 看法 | 公式 |
|---|---|---|
| **計算** | 對應元素相乘再加 | $a_1 b_1 + a_2 b_2$ |
| **幾何** | 投影長度 × 另一向量長度 | $\|\vec{a}\|\|\vec{b}\|\cos\theta$ |
| **對偶** | 向量 $\vec{u}$ 就是一個 2D→1D 的線性變換 | $\vec{u} \cdot \vec{v} = [u_1 \; u_2] \vec{v}$ |

第三層是 3Blue1Brown 這一章最想告訴你的：**點積不只是一個運算技巧，它本質上是一個線性變換**。

未來你在深度學習中會不斷遇到點積：
- **Attention 機制**：Query · Key 就是在問「這兩個向量方向有多像？」
- **全連接層**：每個 neuron 做的就是 $\vec{w} \cdot \vec{x} + b$，也就是把輸入投影到權重方向
- **Loss function**：很多損失函數的核心也是向量間的點積關係